In [6]:
import sqlite3
from langdetect import detect, DetectorFactory, LangDetectException

DB_PATH = "data.nosync/recipes.db"
LANGUAGE_COL = "detected_language"
BATCH_SIZE = 100

# Make langdetect deterministic
DetectorFactory.seed = 0

con = sqlite3.connect(DB_PATH, timeout=5)

cols = {row[1] for row in con.execute("PRAGMA table_info(recipes)")}

if LANGUAGE_COL not in cols:
    print(f"Adding column {LANGUAGE_COL} to recipes table")
    con.execute(f"ALTER TABLE recipes ADD COLUMN {LANGUAGE_COL} TEXT")
    con.commit()



def fetch_batch(batch_size):
    """Fetch up to batch_size rows whose language has not been processed."""
    return con.execute(
        f"""
        SELECT rowid, instructions, ingredients
        FROM recipes
        WHERE {LANGUAGE_COL} IS NULL
        LIMIT ?
        """,
        (batch_size,),
    ).fetchall()


def detect_ingredient_language(ingredients):
    """Detect language, returning NULL when detection is unreliable."""
    if not ingredients or not ingredients.strip():
        return None

    # Ingredient lists that are too short are often impossible to classify.
    if len(ingredients.strip()) < 20:
        return None

    try:
        return detect(ingredients)
    except LangDetectException:
        return None


def update_language(batch):
    updates = []

    for rowid, instructions, ingredients in batch:
        used_string = instructions + ingredients
        language = detect_ingredient_language(used_string)
        
        if language is None:
            language = "lucien"  # Use a placeholder for undetectable languages
        updates.append((language, rowid))

    con.executemany(
        f"""
        UPDATE recipes
        SET {LANGUAGE_COL} = ?
        WHERE rowid = ?
        """,
        updates,
    )
    con.commit()


total_processed = 0
if True:
    try:
        while batch := fetch_batch(BATCH_SIZE):
            update_language(batch)
            total_processed += len(batch)
            print(f"Processed {total_processed} recipes")
    finally:
        con.close()